In [ ]:
# ==============================
# Improved DeBERTa Training Pipeline
# Optimized for Colab A100
# ==============================

# Install deps (run once)
# !pip install -q transformers datasets accelerate scikit-learn pandas numpy

import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# ==============================
# 1. Load + Clean Data
# ==============================

df = pd.read_csv("ExioNAICS.csv")

df = df[["Company Description", "NAICS Code"]].dropna().copy()

df["Company Description"] = df["Company Description"].astype(str).str.strip()

df["NAICS Code"] = (
    df["NAICS Code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# Keep only 6-digit
df = df[df["NAICS Code"].str.len() == 6]

# ==============================
# 2. Drop ultra-rare classes (IMPORTANT)
# ==============================

min_samples = 5
label_counts = df["NAICS Code"].value_counts()
valid_labels = label_counts[label_counts >= min_samples].index

df = df[df["NAICS Code"].isin(valid_labels)].copy()

print("Remaining classes:", df["NAICS Code"].nunique())
print("Remaining rows:", len(df))

# ==============================
# 3. Encode Labels
# ==============================

unique_labels = sorted(df["NAICS Code"].unique())
label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

df["label"] = df["NAICS Code"].map(label2id)

# ==============================
# 4. Tokenizer (LONGER SEQ)
# ==============================

checkpoint = "microsoft/deberta-v3-base"  # upgraded model
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(
        batch["Company Description"],
        truncation=True,
        max_length=MAX_LENGTH
    )

# ==============================
# 5. Metrics (Top-K included)
# ==============================

def top_k_accuracy(logits, labels, k=3):
    top_k_preds = np.argsort(logits, axis=1)[:, -k:]
    return np.mean([labels[i] in top_k_preds[i] for i in range(len(labels))])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "top3_acc": top_k_accuracy(logits, labels, 3),
        "top5_acc": top_k_accuracy(logits, labels, 5),
    }

# ==============================
# 6. Class Weights (CRITICAL)
# ==============================

label_counts = df["label"].value_counts().sort_index().values
class_weights = 1.0 / label_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ==============================
# 7. Custom Trainer with weighted loss
# ==============================

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==============================
# 8. Data Collator (dynamic padding)
# ==============================

data_collator = DataCollatorWithPadding(tokenizer)

# ==============================
# 9. Training Arguments
# ==============================

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=50,
    report_to="none"
)

# ==============================
# 10. Stratified K-Fold
# ==============================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"])):
    print(f"\n===== Fold {fold+1} =====")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_ds = Dataset.from_pandas(train_df[["Company Description", "label"]], preserve_index=False)
    val_ds = Dataset.from_pandas(val_df[["Company Description", "label"]], preserve_index=False)

    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds = val_ds.map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(unique_labels),
        id2label=id2label,
        label2id=label2id
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    metrics = trainer.evaluate()
    print(metrics)
    all_metrics.append(metrics)

# ==============================
# 11. Aggregate Results
# ==============================

def summarize(metric):
    vals = [m[f"eval_{metric}"] for m in all_metrics]
    return np.mean(vals), np.std(vals)

print("\n===== CV RESULTS =====")
for m in ["accuracy", "macro_f1", "weighted_f1", "top3_acc", "top5_acc"]:
    mean, std = summarize(m)
    print(f"{m}: {mean:.4f} ± {std:.4f}")


